Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\Desktop\\pruebas_collab\\datosNarmax\\1pasos_gru_pollution.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [3]:
datos.head()

,pollution,dew,temp,press,wnd_dir,wnd_spd,e
date,,,,,,,
2010-01-02 00:00:00,0.317681,-1.214023,-1.268524,0.329687,-0.380944,-0.464048,NaN
2010-01-02 01:00:00,0.526152,-1.144302,-1.268524,0.329687,-0.380944,-0.446575,NaN
2010-01-02 02:00:00,0.646846,-0.865419,-1.349314,0.426127,-0.380944,-0.429103,NaN
2010-01-02 03:00:00,0.888234,-0.586536,-1.349314,0.522567,-0.380944,-0.393962,NaN
2010-01-02 04:00:00,0.416431,-0.586536,-1.349314,0.522567,-0.380944,-0.376489,NaN


In [4]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Espacio de búsqueda

In [5]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'epochs': hp.choice('epochs', [2 ** i for i in range(3, 9)]),  # Número de épocas de entrenamiento
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [6]:
futuros = 1
pasados  = 12

In [7]:
datosX = []
datosY = []
for i in range(pasados, len(datos) - futuros + 1):
  datosX.append(datos.iloc[i-pasados:i, 0:datos.shape[1]])
  datosY.append(datos.iloc[i+futuros-1:i+futuros, 0])


In [8]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (43788, 12, 7)
Dimensiones de Y: (43788, 1)


In [9]:
print(datosX[0])

[[ 0.31768099 -1.2140229  -1.26852411  0.32968671 -0.38094383 -0.46404777
          nan]
 [ 0.52615226 -1.14430217 -1.26852411  0.32968671 -0.38094383 -0.44657536
          nan]
 [ 0.64684616 -0.86541928 -1.34931411  0.42612698 -0.38094383 -0.42910295
          nan]
 [ 0.88823396 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.39396181
          nan]
 [ 0.41643054 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.3764894
          nan]
 [ 0.09823754 -0.58653639 -1.4301041   0.52256725 -0.38094383 -0.35901698
          nan]
 [ 0.05434885 -0.58653639 -1.4301041   0.61900753 -0.38094383 -0.32387584
          nan]
 [ 0.26282012 -0.58653639 -1.34931411  0.7154478  -0.38094383 -0.2887347
          nan]
 [ 0.21893143 -0.65625711 -1.4301041   0.7154478  -0.38094383 -0.25359356
          nan]
 [ 0.3505975  -0.58653639 -1.34931411  0.81188808 -0.38094383 -0.21845241
          nan]
 [ 0.43837488 -0.58653639 -1.34931411  0.90832835 -0.38094383 -0.15700449
          nan]
 [ 0.57004095 -0.656257

Se dividen nuevamente los conjuntos de datos

In [10]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (30651, 12, 7)
Las dimensiones de testX son:  (8801, 12, 7)
Las dimensiones de valX son:  (4336, 12, 7)


In [11]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (30651, 1)
Las dimensiones de testY son:  (8801, 1)
Las dimensiones de valY son:  (4336, 1)


Se crean métricas para medir desempeño

In [12]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [13]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [14]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(testX.shape[1], testX.shape[2])))
    if (params['layers'] == 1):
      model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(GRU(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(testX, testY, epochs=params['epochs'],
                        validation_split=0.3,
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [15]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=50, trials=trials, rstate=np.random.default_rng(42))

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

193/193 - 24s - 123ms/step - ia: 0.4898 - loss: 0.7025 - mae: 0.6104 - rmse: 0.8140 - smape: 1.1705 - val_ia: 0.3366 - val_loss: 0.2399 - val_mae: 0.3605 - val_rmse: 0.4379 - val_smape: 0.9362

Epoch 2/128                                           

193/193 - 5s - 24ms/step - ia: 0.6134 - loss: 0.5028 - mae: 0.5104 - rmse: 0.6930 - smape: 0.9791 - val_ia: 0.3564 - val_loss: 0.2292 - val_mae: 0.3559 - val_rmse: 0.4324 - val_smape: 0.9342

Epoch 3/128                                           

193/193 - 5s - 25ms/step - ia: 0.6287 - loss: 0.4687 - mae: 0.4915 - rmse: 0.6710 - smape: 0.9376 - val_ia: 0.3683 - val_loss: 0.2202 - val_mae: 0.3523 - val_rmse: 0.4282 - val_smape: 0.9366

Epoch 4/128                                           

193/193 - 5s - 24ms/step - ia: 0.6544 - loss: 0.4177 - mae: 0.4632 - rmse: 0.6306 - smape: 0.8940 - val_ia: 0.3962 - val_loss: 0.2034 - val_mae: 0.3288 - val_rmse: 0.4061 - val_smape: 0.8660

Epoch 

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/16                                                                         

25/25 - 60s - 2s/step - ia: 0.2880 - loss: 0.8908 - mae: 0.6778 - rmse: 0.9319 - smape: 1.4192 - val_ia: 0.5466 - val_loss: 0.2540 - val_mae: 0.3461 - val_rmse: 0.4909 - val_smape: 0.8598

Epoch 2/16                                                                         

25/25 - 8s - 333ms/step - ia: 0.7398 - loss: 0.3111 - mae: 0.3774 - rmse: 0.5608 - smape: 0.7152 - val_ia: 0.6477 - val_loss: 0.1664 - val_mae: 0.2836 - val_rmse: 0.4015 - val_smape: 0.7432

Epoch 3/16                                                                         

25/25 - 5s - 181ms/step - ia: 0.8014 - loss: 0.1856 - mae: 0.2933 - rmse: 0.4279 - smape: 0.6118 - val_ia: 0.7112 - val_loss: 0.1251 - val_mae: 0.2423 - val_rmse: 0.3473 - val_smape: 0.6599

Epoch 4/16                                                                         

25/25 - 5s - 218ms/step - ia: 0.8500 - loss: 0.1225 - mae: 0.2324 - rmse: 0.3436 - smape:

...

770/770 - 7s - 9ms/step - ia: 0.7863 - loss: 0.1879 - mae: 0.2629 - rmse: 0.3542 - smape: 0.5725 - val_ia: 0.4459 - val_loss: 0.0634 - val_mae: 0.1567 - val_rmse: 0.1963 - val_smape: 0.4952

Epoch 2/16                                                                           

770/770 - 3s - 4ms/step - ia: 0.8787 - loss: 0.0636 - mae: 0.1540 - rmse: 0.2164 - smape: 0.3758 - val_ia: 0.5167 - val_loss: 0.0542 - val_mae: 0.1334 - val_rmse: 0.1727 - val_smape: 0.4334

Epoch 3/16                                                                           

770/770 - 3s - 4ms/step - ia: 0.8889 - loss: 0.0593 - mae: 0.1453 - rmse: 0.2068 - smape: 0.3530 - val_ia: 0.5224 - val_loss: 0.0534 - val_mae: 0.1312 - val_rmse: 0.1706 - val_smape: 0.4276

Epoch 4/16                                                                           

770/770 - 3s - 4ms/step - ia: 0.8905 - loss: 0.0580 - mae: 0.1427 - rmse: 0.2040 - smape: 0.3490 - val_ia: 0.5331 - val_loss: 0.0518 - val_mae: 0.1278 - val_rmse: 0.1

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



97/97 - 7s - 68ms/step - ia: 0.8412 - loss: 0.1278 - mae: 0.2382 - rmse: 0.3394 - smape: 0.4997 - val_ia: 0.7914 - val_loss: 0.0588 - val_mae: 0.1409 - val_rmse: 0.2118 - val_smape: 0.4497

Epoch 2/128                                                                          

97/97 - 3s - 26ms/step - ia: 0.8774 - loss: 0.0845 - mae: 0.1882 - rmse: 0.2824 - smape: 0.4060 - val_ia: 0.7704 - val_loss: 0.0616 - val_mae: 0.1502 - val_rmse: 0.2178 - val_smape: 0.4592

Epoch 3/128                                                                          

97/97 - 2s - 25ms/step - ia: 0.8824 - loss: 0.0779 - mae: 0.1807 - rmse: 0.2705 - smape: 0.3979 - val_ia: 0.7799 - val_loss: 0.0605 - val_mae: 0.1440 - val_rmse: 0.2126 - val_smape: 0.4547

Epoch 4/128                                                                          

97/97 - 2s - 24ms/step - ia: 0.8833 - loss: 0.0760 - mae: 0.1790 - rmse: 0.2678 - smape: 0.3990 - val_ia: 0.7798 - val_loss: 0.0576 - val_mae: 0.1422 - val_rmse: 0.2094 

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



49/49 - 5s - 108ms/step - ia: 0.3885 - loss: 2.0080 - mae: 1.0171 - rmse: 1.4064 - smape: 1.1851 - val_ia: 0.3502 - val_loss: 1.1361 - val_mae: 0.8032 - val_rmse: 0.9844 - val_smape: 1.0779

Epoch 2/32                                                                           

49/49 - 1s - 10ms/step - ia: 0.3903 - loss: 1.9547 - mae: 1.0021 - rmse: 1.3986 - smape: 1.1867 - val_ia: 0.3539 - val_loss: 1.1004 - val_mae: 0.7850 - val_rmse: 0.9667 - val_smape: 1.0694

Epoch 3/32                                                                           

49/49 - 1s - 13ms/step - ia: 0.3880 - loss: 1.9003 - mae: 0.9837 - rmse: 1.3577 - smape: 1.1859 - val_ia: 0.3573 - val_loss: 1.0662 - val_mae: 0.7674 - val_rmse: 0.9496 - val_smape: 1.0611

Epoch 4/32                                                                           

49/49 - 0s - 8ms/step - ia: 0.3879 - loss: 1.8671 - mae: 0.9738 - rmse: 1.3626 - smape: 1.1936 - val_ia: 0.3604 - val_loss: 1.0341 - val_mae: 0.7508 - val_rmse: 0.9333 

In [16]:
print(best)

{'activation': 1, 'batch': 2, 'dropout': 0.1, 'epochs': 4, 'layers': 1.0, 'learning_rate': 0.00027756193103546005, 'units': 4}
